# Estimating the Causal Effect of a Policy Using Difference-in-Differences

Difference-in-Differences (DID) is one of the most commonly used quasi-experimental designs in social science. The intuition is straightforward: a policy takes effect at a specific time for only some units (the treated group), so subtracting the contemporaneous change in the control group from the change in the treated group before and after the policy isolates the common time trend, yielding the policy's **average treatment effect on the treated (ATT)**.

Whether DID can be interpreted as causal hinges on a critical assumption—**parallel trends**: absent the policy, the treated and control groups' outcomes would have evolved along parallel trajectories. This assumption cannot be directly tested, but we can indirectly examine it using pre-treatment periods' "pre-trends." This tutorial walks through a complete identification chain: load the panel → declare the design → test parallel trends → estimate ATT → unpack dynamic effects → robustness checks → plot results.

We use `socialverse` to complete the workflow. It is an analytics library built for social science that registers each method in a function registry and at runtime verifies whether "the prerequisites for this step are in place," while recording each step in a reproducible evidence chain—you will see it at the end. For methodological background, see Roth et al. (2023) *What's Trending in Difference-in-Differences?* and documentation for R packages such as `fixest` and `did`.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import socialverse as sv
from socialverse import datasets as ds

## Loading Panel Data

We use a built-in synthetic panel: 40 firms × 8 years (2010–2017). Half of the firms are covered by a policy in 2015; the true ATT is approximately −0.8. The data is designed with clean parallel pre-trends, which makes it easy for us to see each step of the identification chain clearly.

The panel is in long format (one row per firm–year combination): `firm_id` is the unit, `year` is the time index, `treat_post` flags observations that have been treated, `first_treated` is the year each firm was first treated, and `y` is the outcome variable.

In [2]:
df = ds.load_did_panel(att=-0.8)
print("面板维度:", df.shape)
df.head()

面板维度: (320, 8)


,firm_id,year,treat,post,treat_post,first_treated,y,x1
0,0,2010,1,0,0,2015,1.6221,1.5139
1,0,2011,1,0,0,2015,2.8355,0.7813
2,0,2012,1,0,0,2015,2.5744,-0.3139
3,0,2013,1,0,0,2015,3.2232,1.9603
4,0,2014,1,0,0,2015,3.5321,1.3151


## Declaring the Research Design

The first step of analysis is not to run a regression, but to clarify which variables play which roles. `declare_design` registers the panel unit ID, time, treatment indicator, and treatment start year into the research state; all subsequent causal functions read the design from there, avoiding the need to pass parameters repeatedly.

In [3]:
st = sv.StudyState()
st.write("estimand", "target", "ATT")   # 我们要估的是平均处理效应,而非单纯的相关
st.write("variables", "outcome", "y")   # 结果变量

sv.pp.ingest(st, data=df, name="policy_panel")
sv.pp.declare_design(
    st,
    panel_id="firm_id",
    time="year",
    treatment="treat_post",
    first_treated="first_treated",
)
st.design

Slot(panel_id, time, treatment, first_treated)

## Testing Parallel Trends

This is the gatekeeper of the entire chain. `parallel_trends` estimates a full event study (unit fixed effects + time fixed effects), then conducts a joint test on all **pre-treatment** relative-period coefficients. The null hypothesis is "all pre-treatment period coefficients equal zero," which is to say the two groups' trends are parallel before treatment.

If `p > 0.05`, we fail to reject parallel trends and the identification assumption holds; if `p` is very small, the pre-trends have already diverged, and even if we produce coefficients downstream, we should not call them "causal."

In [4]:
sv.tl.parallel_trends(st)

pt = st.diagnostics["pretrend"]
print("平行趋势判定:", st.identification["parallel_trends"])
print(f"联合 F = {pt['joint_F']:.2f}   p = {pt['p_value']:.3f}   (前导期数 = {pt['n_pre']})")

平行趋势判定: pass
联合 F = 0.47   p = 0.755   (前导期数 = 4)


`p` value is clearly greater than 0.05—the joint test of pre-treatment period coefficients is not significant, parallel trends hold, and we can proceed to estimation. This determination is recorded in the research state and becomes a prerequisite for the next step, `did`.

## Estimating ATT

Now we can estimate. `did` fits `y ~ treat_post + unit fixed effects + time fixed effects` and computes robust standard errors clustered at the `firm_id` level (inference on treatment effects typically requires clustering at the unit level). It simultaneously reads the parallel trends determination from the previous step into the conclusion: if passed, it is annotated as "causal ATT"; if not passed, it is downgraded to "association, not causal."

In [5]:
sv.tl.did(st)

m = st.models["did"]
print(f"ATT   = {m['att']:.3f}")
print(f"95%CI = [{m['ci'][0]:.3f}, {m['ci'][1]:.3f}]")
print(f"SE    = {m['se']:.3f}   (聚类于 {m['n_clusters']} 家企业)")
print(f"p     = {m['p']:.2e}")
print("结论  :", m["note"])

ATT   = -0.731
95%CI = [-0.931, -0.531]
SE    = 0.102   (聚类于 40 家企业)
p     = 8.15e-13
结论  : 因果 ATT(平行趋势通过)


The estimated ATT ≈ −0.73, the 95% confidence interval excludes 0 and covers the true value −0.8. The policy causes the outcome variable to decrease significantly.

## Unpacking Dynamic Effects: Event Study

ATT is an "average." The event study decomposes it across relative treatment periods: using the period before treatment (−1) as the reference, it reports coefficients for each relative period. Pre-treatment coefficients (< 0) let us examine once more whether pre-trends hug the zero line; post-treatment coefficients (≥ 0) characterize how the effect evolves over time after the policy takes effect.

In [6]:
import pandas as pd

sv.tl.event_study(st)
es = st.models["event_study"]
pd.DataFrame(
    [{"相对时点": int(k), "系数": round(v[0], 3), "SE": round(v[1], 3)} for k, v in es["coefs"].items()]
)

,相对时点,系数,SE
0,-5,0.024,0.176
1,-4,-0.112,0.165
2,-3,-0.226,0.188
3,-2,-0.145,0.226
4,-1,0.000,0.000
5,0,-0.888,0.154
6,1,-0.786,0.162
7,2,-0.794,0.175


## Robustness: Sensitivity of Standard Errors to Specification

Once the point estimate is obtained, a routine check is whether standard errors are stable across different variance specifications. The robustness matrix produced by `did` compares standard errors across three specifications—classical (homoskedastic), heteroskedasticity-robust (HC1), and clustered by firm. Clustered SE is typically largest and most credible because it allows for within-firm correlation across years.

In [7]:
print(st.diagnostics["robustness"])

{'specs': [{'spec': 'classical', 'att': -0.7307386666666635, 'se': 0.09400723212228163, 'p': 1.581384999242552e-13}, {'spec': 'HC1_robust', 'att': -0.7307386666666635, 'se': 0.09246780376686374, 'p': 2.7308339489037207e-15}, {'spec': 'cluster_panel', 'att': -0.7307386666666635, 'se': 0.1020793428061824, 'p': 8.154308083646252e-13}], 'note': 'ATT 在 classical / HC1 / panel-cluster SE 下对比'}


## Publication-Ready Plots

Once results are ready, `socialverse` plotting functions read directly from the research state and produce plots without manual data wrangling. We create two plots here: a forest plot of the ATT and an event study plot of the dynamic effects.

In [8]:
sv.pl.forest(st, out="fig_forest.png", title="政策 ATT · 点估计 ± 95% CI")
sv.pl.event_study_plot(st, out="fig_eventstudy.png")
print("图已保存:fig_forest.png, fig_eventstudy.png")

图已保存:fig_forest.png, fig_eventstudy.png


**Forest plot**—single ATT coefficient and 95% confidence interval, with a dashed line at zero for reference:

![Forest plot](fig_forest.png)

**Event study plot**—lead periods hug the zero line (parallel trends), and after period 0 jumps to approximately −0.8 and persists:

![Event study plot](fig_eventstudy.png)

## A Reproducible Evidence Chain

Finally, take a look at the key difference between `socialverse` and ordinary estimation scripts. After running the entire chain, the research state automatically accumulates a **provenance ledger**: which function was used at each step, what it consumed, and what it produced. This ledger makes the analysis traceable and reproducible—in social science, "from which step and which data the conclusions come" is often as important as the conclusions themselves.

In [9]:
print(st.summary())

StudyState
  sources: datasets, dataset_name, dataset_meta
  design: panel_id, time, treatment, first_treated
  variables: outcome
  estimand: target
  identification: parallel_trends
  models: twfe, did, event_study
  diagnostics: pretrend, robustness
  artifacts: figures
  provenance: 7 step(s)


## Summary

We have completed a standard DID identification chain: declare design → test parallel trends → estimate ATT → unpack dynamic effects → check robustness → plot results. It aligns with the workflows of `pyfixest` and R's `fixest` (high-dimensional fixed effects + clustered robust SE) and `did` (Callaway–Sant'Anna).

Compared to pure estimation tools, we add two things here: parallel trends is a **real gatekeeper** (if it fails, conclusions automatically downgrade rather than silently giving you a coefficient), and an evidence chain running throughout. The next tutorial [03_complex_survey](03_complex_survey.ipynb) turns to design-weighted inference for complex survey samples.